# 🧪 GELU Activation Experiment

**Goal**: Test if biologically-plausible GELU activation improves NER performance.

**What is GELU?**
- Gaussian Error Linear Unit
- Smooth, differentiable everywhere (unlike ReLU)
- Used in BERT, GPT (proven effective)
- More biologically plausible (stochastic neuron activation)

**This notebook is STANDALONE** - fresh Colab session, no previous runs needed!

## 1. Check GPU

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ No GPU - training will be slow!")

## 2. Clone Repository

In [ ]:
# Clone from GitHub
!git clone https://github.com/bhushan1729/olfaction-inspired-ner.git
%cd olfaction-inspired-ner

## 3. Install Dependencies

In [ ]:
!pip install -q torch numpy scikit-learn seqeval matplotlib seaborn pandas tqdm tensorboard requests

## 4. Download GloVe Embeddings

In [ ]:
import os

if not os.path.exists('./data/glove.6B.300d.txt'):
    print("Downloading GloVe embeddings (takes ~5 min)...")
    !mkdir -p data
    !wget -q http://nlp.stanford.edu/data/glove.6B.zip -O data/glove.6B.zip
    !unzip -q data/glove.6B.zip -d data/
    !rm data/glove.6B.zip
    print("✓ GloVe downloaded")
else:
    print("✓ GloVe already exists")

## 5. Fix Python Path

In [ ]:
import sys
sys.path.append('/content/olfaction-inspired-ner')
print("✓ Path configured")

## 6. Train with GELU Activation

This will:
- Download CoNLL-2003 automatically
- Train olfactory model with **GELU** activation in receptors
- Takes ~20-30 minutes on GPU

In [ ]:
# Train with GELU
!python src/train.py \
  --config config/tuning_experiments.yaml \
  --experiment activation_gelu \
  --save_dir results/tuning/gelu

## 7. View Results

In [ ]:
import json

# Load results
with open('results/tuning/gelu/results.json') as f:
    results = json.load(f)

print("\n" + "="*60)
print(" "*15 + "GELU EXPERIMENT RESULTS")
print("="*60)

test = results['test']
print(f"\nTest F1:        {test['f1']:.4f}")
print(f"Precision:      {test['precision']:.4f}")
print(f"Recall:         {test['recall']:.4f}")

print("\nPer-entity F1 scores:")
print("-" * 60)
for entity, score in test['per_entity'].items():
    if entity in ['LOC', 'MISC', 'ORG', 'PER']:
        print(f"  {entity:<10} {score:.4f}")

print("\n" + "="*60)
print("\n✓ Training complete!")
print("\nShare these results with the comparison task.")

## 8. Analyze Receptor Activations (Optional)

In [ ]:
# Analyze what receptors learned with GELU
import torch
from src.data.dataset import prepare_data
from src.model.olfactory_ner import create_olfactory_ner
from src.analysis.visualize import analyze_receptor_activations
import yaml

# Load config
with open('config/tuning_experiments.yaml') as f:
    config = yaml.safe_load(f)

exp_config = config['activation_gelu']
exp_config.update(config['data'])

# Load data
print("Loading data...")
_, _, test_loader, vocab_info = prepare_data(
    data_dir='./data/raw',
    batch_size=32
)

# Load model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
checkpoint = torch.load(
    'results/tuning/gelu/best_model.pt',
    map_location=device,
    weights_only=False
)

vocab_size = len(vocab_info['word2idx'])
num_tags = len(vocab_info['label2idx'])

model = create_olfactory_ner(vocab_size, num_tags, exp_config)
model.load_state_dict(checkpoint['model_state_dict'])
model = model.to(device)

# Analyze
print("\nAnalyzing GELU receptor activations...")
analyze_receptor_activations(
    model,
    test_loader,
    vocab_info,
    device,
    save_dir='./analysis_gelu'
)

print("\n✓ Analysis complete! Check analysis_gelu/ folder")

## 9. Display Visualizations (Optional)

In [ ]:
from IPython.display import Image, display

print("📊 Receptor Activation Heatmap (GELU):")
display(Image('analysis_gelu/receptor_heatmap.png'))

print("\n📊 t-SNE Visualization (GELU):")
display(Image('analysis_gelu/glomeruli_tsne.png'))

## 10. Download Results

In [ ]:
# Package everything
!zip -r gelu_experiment_results.zip results/tuning/gelu/ analysis_gelu/

# Download
from google.colab import files
files.download('gelu_experiment_results.zip')

print("\n✓ Results downloaded!")

---

## 📋 Summary

**What you have now:**
- ✅ GELU model trained on CoNLL-2003
- ✅ Test F1 score (shown above)
- ✅ Per-entity performance
- ✅ Receptor activation visualizations
- ✅ All results downloaded

**Next step:**
Provide these results to compare with ReLU baseline!